## Building chatbot with Multiple tools using Langgraph

#### Aim
Create a chatbot with tool capabilities from Arxiv, Wikipedia search and some custom function. 

Langchain provide us some pre-build [tools](https://python.langchain.com/docs/integrations/tools/). We will use:
1. [Arxiv](https://python.langchain.com/docs/integrations/tools/arxiv)
2. [Wikipedia](https://python.langchain.com/docs/integrations/tools/wikipedia)
3. [Tavily](https://python.langchain.com/docs/integrations/tools/tavily_search/)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.environ['GROQ_API_KEY']
assert GROQ_API_KEY
print(f'GROQ_API_KEY: {GROQ_API_KEY[:3]}**{GROQ_API_KEY[-3:]}')

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
assert OPENAI_API_KEY
print(f'OPENAI_API_KEY: {OPENAI_API_KEY[:3]}**{OPENAI_API_KEY[-3:]}')

TAVILY_API_KEY = os.environ['TAVILY_API_KEY']
assert TAVILY_API_KEY
print(f'TAVILY_API_KEY: {TAVILY_API_KEY[:3]}**{TAVILY_API_KEY[-3:]}')

GROQ_API_KEY: gsk**qNU
OPENAI_API_KEY: sk-**tIA
TAVILY_API_KEY: tvl**66J


In [2]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun, TavilySearchResults
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper


In [3]:
arxiv_api = ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=500)
arxiv_query_run = ArxivQueryRun(api_wrapper=arxiv_api)
arxiv_query_run.name

'arxiv'

In [4]:
arx_resp = arxiv_query_run.invoke(input='Attention is all you need')
arx_resp

"Published: 2024-07-22\nTitle: Attention Is All You Need But You Don't Need All Of It For Inference of Large Language Models\nAuthors: Georgy Tyukin, Gbetondji J-S Dovonon, Jean Kaddour, Pasquale Minervini\nSummary: The inference demand for LLMs has skyrocketed in recent months, and serving\nmodels with low latencies remains challenging due to the quadratic input length\ncomplexity of the attention layers. In this work, we investigate the effect of\ndropping MLP and attention layers at inference time o"

In [5]:
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500)
wiki_query_run = WikipediaQueryRun(api_wrapper=wiki_api)
wiki_query_run.name

'wikipedia'

In [ ]:
wiki_response = wiki_query_run.invoke('Levski Sofia')

'Page: PFC Levski Sofia\nSummary: PFC Levski Sofia (Bulgarian: ПФК Левски София) is a Bulgarian professional association football club based in Sofia, which competes in the First League, the top division of the Bulgarian football league system. The club was founded on 24 May 1914 by a group of high school students, and is named after Vasil Levski, a Bulgarian revolutionary renowned as the national hero of the country.\nLevski have won a total of 74 trophies, including 26 national championships, 26 '

In [9]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(top_k_results=2)
tav_resp = tavily_search.invoke('Deformable Detr')
tav_resp

{'query': 'Deformable Detr',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://cenk-bircanoglu.medium.com/paper-review-deformable-transformers-for-end-to-end-object-detection-ed0a452f775f',
   'title': 'Paper Review: Deformable Transformers for End-to-End ...',
   'content': 'To this end, Deformable DETR architecture enhances the DETR model by incorporating deformable attention mechanisms and employing multi-scaled features. In summary, Deformable DETR extends the capabilities of DETR by introducing deformable attention, which enhances spatial adaptability and improves the model’s performance in tasks like object detection.',
   'score': 0.92359936,
   'raw_content': None},
  {'url': 'https://huggingface.co/docs/transformers/v4.26.1/en/model_doc/deformable_detr',
   'title': 'Deformable DETR',
   'content': 'Deformable DETR mitigates the slow convergence issues and limited feature spatial resolution of the original DETR by leveraging a new defo

#### Combine all the tools in a list

In [13]:
from langchain_core.tools import BaseTool

tools = [arxiv_query_run, wiki_query_run, tavily_search]

for tool in tools:
    assert isinstance(tool, BaseTool)

#### Initialize Groq Chat

In [14]:
from langchain_groq.chat_models import ChatGroq

assert GROQ_API_KEY

model_name = 'Gemma2-9b-It'
chat_groq = ChatGroq(model=model_name)
groq_with_tools = chat_groq.bind_tools(tools=tools)
groq_with_tools

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x116cf8cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x116cf96d0>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'arxiv', 'description': 'A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'search query to look up', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical even

In [17]:
from langchain_core.messages import AIMessage, HumanMessage

groq_input = [HumanMessage(content='What is the news about Ukraine-Russia war for 2 September 2025?')]

groq_response = groq_with_tools.invoke(input=groq_input)

In [19]:
groq_response.tool_calls

[{'name': 'tavily_search',
  'args': {'end_date': '2025-09-02',
   'query': 'news Ukraine-Russia war 2 September 2025',
   'start_date': '2025-09-02'},
  'id': '36ewjtrh5',
  'type': 'tool_call'}]

### Create the entire chatbot with Langgraph

In [20]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [32]:
from IPython.display import display, Image
from langchain_core.runnables.graph import MermaidDrawMethod
from langgraph.graph import StateGraph, START, END

from langgraph.prebuilt import tools_condition, ToolNode

def tool_calling_llm(state: State):
    return {'messages': [groq_with_tools.invoke(state['messages'])]}

graph_builder = StateGraph(State)

# Nodes definition
graph_builder.add_node('LLM_call', tool_calling_llm)
graph_builder.add_node('tools', ToolNode(tools=tools))

# Edges definition
graph_builder.add_edge(START, 'LLM_call')
graph_builder.add_conditional_edges('LLM_call', tools_condition)
graph_builder.add_edge('tools', END)

# Compile 
graph = graph_builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [37]:
input_messages = [HumanMessage('1706.03762')]
state = {'messages': input_messages}
result = graph.invoke(state)

In [38]:
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

1706.03762
================================== Ai Message ==================================
Tool Calls:
  arxiv (tn106hrkc)
 Call ID: tn106hrkc
  Args:
    query: 1706.03762
================================= Tool Message =================================
Name: arxiv

Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks in an encoder-decoder configuration. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer, base


In [42]:
input_messages = [HumanMessage('Provide me the top news for Russia-Ukraine conflict for 02 September 2025')]
state = {'messages': input_messages}
result = graph.invoke(state)

In [43]:
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

Provide me the top news for Russia-Ukraine conflict for 02 September 2025
================================== Ai Message ==================================
Tool Calls:
  tavily_search (cs2n9w2dv)
 Call ID: cs2n9w2dv
  Args:
    end_date: 2025-09-02
    query: Russia-Ukraine conflict top news 02 Sep 2025
    start_date: 2025-09-02
================================= Tool Message =================================
Name: tavily_search

{'error': ValueError('Error 400: start_date and end_date cannot be the same')}
